## Nguồn gốc của Dataset

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import PowerTransformer
import category_encoders as ce
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor

import os
# Giảm hiện tượng phân mảnh VRAM
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Giải phóng bộ nhớ đệm trước khi train
import torch
torch.cuda.empty_cache()
# Load the dataset
file_path_train = './menor%bots_2/35bot/2000000samples2000epochs.csv'
file_path_test = './menor%bots_1/5bot/2000000samples2000epochs.csv'
df = pd.read_csv(file_path_train)
df_test = pd.read_csv(file_path_test)

## Preprocessing

In [14]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, QuantileTransformer

class Preprocessor:
    def __init__(self):
        # 1. Khởi tạo các công cụ (Chưa có dữ liệu)
        self.ohe_trigger = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        self.ohe_vm = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        self.scaler = MinMaxScaler(feature_range=(0, 1))
        self.qt = QuantileTransformer(output_distribution='uniform')
        
        # Lưu trữ danh sách các cột phân phối để dùng lại lúc Test
        self.smooth_cols = []
        self.skewed_cols = []

    def engineer_features(self, df_raw):
        """Hàm dùng chung để trích xuất thời gian và IP"""
        df = df_raw.copy()

        # Xử lý thời gian
        df['timestamp'] = pd.to_datetime(df['timestamp'], format='ISO8601', utc=True)
        df['hourOfDay'] = df['timestamp'].dt.hour
        df['dayOfWeek'] = df['timestamp'].dt.dayofweek

        # BẮT BUỘC: Sort theo thời gian
        df = df.sort_values(by=['IP', 'timestamp']).reset_index(drop=True)

        # Đặt timestamp làm index (Hàm rolling theo thời gian yêu cầu điều này)
        df = df.set_index('timestamp')

        # Xử lý IP - Rolling Windows
        df['ipMin'] = df.groupby('IP')['IP'].rolling('1min').count().reset_index(level=0, drop=True)
        df['ipHour'] = df.groupby('IP')['IP'].rolling('1h').count().reset_index(level=0, drop=True)
        df['ipDay'] = df.groupby('IP')['IP'].rolling('1d').count().reset_index(level=0, drop=True)

        # Trả timestamp về lại thành cột để các hàm sau (như fit_transform) không bị lỗi
        df = df.reset_index()

        # Nếu để yên FunctionId, model có nguy cơ tạo ra các quy tắc ngớ ngẩn như:␣"Nếu FunctionId > 25 thì là Botnet"
        df['FunctionId'] = df['FunctionId'].astype('category')

        cols_to_drop = ['IP', 'timestamp']

        df = df.drop(columns = cols_to_drop)
        
        return df

    def fit_transform(self, df_raw):
        """CHỈ DÙNG CHO TẬP TRAIN"""
        print("Đang huấn luyện cỗ máy Preprocessor với tập Train...")
        df = df_raw
        cols_to_drop = ['IP', 'functionTrigger', 'vmcategory', 'timestamp', 'Id']
        
        # 1. Học One-Hot Encoding
        encoded_trigger = self.ohe_trigger.fit_transform(df[['functionTrigger']])
        df_trigger = pd.DataFrame(encoded_trigger, columns=self.ohe_trigger.get_feature_names_out())
        
        encoded_vm = self.ohe_vm.fit_transform(df[['vmcategory']])
        df_vm = pd.DataFrame(encoded_vm, columns=self.ohe_vm.get_feature_names_out())
        
        df = pd.concat([df, df_trigger, df_vm], axis=1).drop(columns=['Id'], errors='ignore')
        
        # 2. Học phân phối để chia Scaling
        # Bỏ qua các cột Category/OHE/Target để tìm cột số thật sự
        numeric_cols = [col for col in df.columns if col not in cols_to_drop and col != 'bot' and '_' not in col and col != 'FunctionId']
        
        for col in numeric_cols:
            if abs(df[col].skew()) > 1.0:
                self.skewed_cols.append(col)
            else:
                self.smooth_cols.append(col)
                
        # 3. Học cách Scaling
        if self.smooth_cols:
            df[self.smooth_cols] = self.scaler.fit_transform(df[self.smooth_cols])
        if self.skewed_cols:
            df[self.skewed_cols] = self.qt.fit_transform(df[self.skewed_cols])
            
        # 4. Rời rạc hóa (Discretize * 50)
        for col in numeric_cols:
            df[col] = np.round(df[col] * 50).astype(int)
        
        return df

    def transform(self, df_raw):
        """CHỈ DÙNG CHO TẬP TEST (ÁP DỤNG LUẬT ĐÃ HỌC, KHÔNG HỌC THÊM)"""
        print("Đang xử lý tập Test với các luật đã học...")
        df = df_raw
        
        # 1. Áp dụng OHE
        encoded_trigger = self.ohe_trigger.transform(df[['functionTrigger']])
        df_trigger = pd.DataFrame(encoded_trigger, columns=self.ohe_trigger.get_feature_names_out())
        
        encoded_vm = self.ohe_vm.transform(df[['vmcategory']])
        df_vm = pd.DataFrame(encoded_vm, columns=self.ohe_vm.get_feature_names_out())
        
        df = pd.concat([df, df_trigger, df_vm], axis=1).drop(columns=['Id'], errors='ignore')
        
        # 2. Áp dụng Scaling cũ
        if self.smooth_cols:
            df[self.smooth_cols] = self.scaler.transform(df[self.smooth_cols])
        if self.skewed_cols:
            df[self.skewed_cols] = self.qt.transform(df[self.skewed_cols])
            
        # 3. Rời rạc hóa
        numeric_cols = self.smooth_cols + self.skewed_cols
        for col in numeric_cols:
            df[col] = np.round(df[col] * 50).astype(int)

        return df

    def packing(self, df):
        # Đóng gói
        cols_to_drop = ['functionTrigger', 'vmcategory', 'Id']
        y = df['bot'].astype(int).values
        X_df = df.drop(columns=[c for c in cols_to_drop if c in df.columns] + ['bot'], errors='ignore')
        # ---> THÊM DÒNG NÀY ĐỂ LƯU TÊN CỘT CHO XGBOOST <---
        feature_names = X_df.columns.tolist()
        X = X_df.astype(int).values
        
        return X, y, feature_names

In [11]:
# Khởi tạo
preprocessor = Preprocessor()

## Feature Profiling
For each feature:
- Observe overall data distribution
- Consider value ordering (rank behavior)
- Check how values are spread (dense, skewed, or repetitive)

In [12]:
def feature_profiling(df, name="TRAIN"):
    # 1. Định nghĩa các mốc phân vị
    q_levels = np.arange(0, 1.01, 0.05)
    features = [col for col in df.columns if col != 'bot' and col != 'Id' and '_' not in col] 
    
    print(f"FEATURE PROFILING cho Dataset: {name}")
    
    for col in features:
        print(f"\n" + "="*50)
        print(f"FEATURE PROFILING: {col}")
        print("="*50)

        # KIỂM TRA: Nếu là cột số (int, float)
        if pd.api.types.is_numeric_dtype(df[col]):
            print(f"Loại dữ liệu: Numeric ({df[col].dtype})")
            print(f"Unique values: {df[col].nunique()}")
        
            # --- CHECK SPREAD (Sự phân tán) ---
            skewness = df[col].skew()
            kurtosis = df[col].kurtosis()
            # Lấy phần trăm của giá trị xuất hiện nhiều nhất
            most_freq_val_pct = df[col].value_counts(normalize=True).iloc[0] * 100
        
            print("\n--- SPREAD & RANK ANALYSIS ---")
            print(f"1. Độ lệch (Skewness): {skewness:.2f} -> {'Bị lệch rất nặng (Nên cân nhắc Transform)' if abs(skewness) > 1 else 'Phân phối tương đối ổn'}")
            print(f"2. Độ lặp lại (Repetitive): {most_freq_val_pct:.2f}% dữ liệu nằm ở 1 giá trị duy nhất.")
        
            # --- RANK BEHAVIOR (Tương quan Spearman với target) ---
            # Giả định cột 'bot' của bạn là dạng số (0 và 1)
            if pd.api.types.is_numeric_dtype(df['bot']):
                spearman_corr = df[col].corr(df['bot'], method='spearman')
                print(f"3. Spearman Rank Correlation (với 'bot'): {spearman_corr:.4f}")
            print("------------------------------\n")

            # Tính toán quantiles
            quantiles = df[col].quantile(q_levels)
            q_df = pd.DataFrame({
                'Percentile': [f"{int(q*100)}%" for q in q_levels],
                'Value': quantiles.values
            })
            print("--- QUANTILES ---")
            print(q_df.to_string(index=False))
        
            # Vẽ Histplot cho dữ liệu số
            # plt.figure(figsize=(10, 4))
            # sns.histplot(df[col], kde=True, color='teal')
            # plt.axvline(df[col].median(), color='red', linestyle='--', label=f'Median: {df[col].median():.2f}')
            # plt.title(f"Numerical Distribution: {col}")
            # plt.legend()
            # plt.show()

        # KIỂM TRA: Nếu là cột phân loại (category, object)
        else:
            print(f"Loại dữ liệu: Categorical ({df[col].dtype})")
            print(f"Unique values: {df[col].nunique()}")
        
            # Với Categorical, mình cũng check độ lặp lại để xem có bị mất cân bằng không
            most_freq_val_pct = df[col].value_counts(normalize=True).iloc[0] * 100
            print(f"Độ lặp lại: Giá trị phổ biến nhất chiếm {most_freq_val_pct:.2f}% tổng dữ liệu.")
        
            print("\nTop 10 giá trị xuất hiện nhiều nhất:")
            print(df[col].value_counts().head(10))

            # Vẽ Countplot cho dữ liệu phân loại
            # plt.figure(figsize=(10, 4))
            # top_10 = df[col].value_counts().head(10).index
            # sns.countplot(
            #     data=df[df[col].isin(top_10)], 
            #     x=col, 
            #     hue=col,
            #     order=top_10, 
            #     palette='viridis',
            #     dodge=False,
            #     legend=False
            # )
            # plt.title(f"Top 10 Categories: {col}")
            # plt.xticks(rotation=45) 
            # plt.show()

In [13]:
# Tạo feature mới
df = preprocessor.engineer_features(df)
df_test = preprocessor.engineer_features(df_test)

ValueError: passed window 1hour is not compatible with a datetimelike index

In [ ]:
# Feature Profiling
feature_profiling(df, "TRAIN")
feature_profiling(df_test, "TEST")

## Feature Importance + Train XGBoost (ML)

In [ ]:
import xgboost as xgb
import matplotlib.pyplot as plt
import pandas as pd
from xgboost import plot_importance

columns_to_drop = ['dayOfWeek', "hourOfDay"]

df = df.drop(columns = columns_to_drop, errors='ignore')
df_test = df.drop(columns = columns_to_drop, errors='ignore')

X_train, y_train, feature_names = preprocessor.packing(df)
X_test_ml, y_test_ml, _ = preprocessor.packing(df_test)

print("--- ĐANG HUẤN LUYỆN XGBOOST ---")
# 1. Tính trọng số cho XGBoost (Tương tự pos_weight của PyTorch)
ratio = (y_train == 0).sum() / (y_train == 1).sum()

# 2. Khởi tạo và Train
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=ratio,
    random_state=42,
    n_jobs=-1,
    enable_categorical=True,
)
xgb_model.fit(X_train, y_train)

# 3. Vẽ Feature Importance
importance_scores = xgb_model.get_booster().get_score(importance_type='gain')

# Ghép tên cột từ preprocessor vào điểm số
feat_importance = {}
for i, name in enumerate(feature_names):
    feat_key = f'f{i}'
    if feat_key in importance_scores:
        feat_importance[name] = importance_scores[feat_key]

importance_df = pd.DataFrame(
    list(feat_importance.items()), 
    columns=['Feature', 'Gain Score']
).sort_values(by='Gain Score', ascending=False)

# Liệt kê
print("Bảng trọng số các cột ảnh hưởng nhất:")
print(importance_df)

## Model Input (Train Transformer - DL)

Feed the matrix into a deep learning model (e.g., attention-based)

In [ ]:
# Preprocessing (Scaling, rời rạc ...) => 2 tập train và test
df = preprocessor.fit_transform(df)
df_test = preprocessor.transform(df_test)

In [ ]:
X_train, y_train, _ = preprocessor.packing(df)
X_test_dl, y_test_dl, _ = preprocessor.packing(df_test)

### Kiến trúc Transformer

In [ ]:
import torch
import torch.nn as nn

class TabularTokenizer(nn.Module):
    def __init__(self, num_features, vocab_size, d, k):
        super().__init__()
        # 1. Value Embedding (Map giá trị normalized integer -> vector d)
        self.value_embed = nn.Embedding(num_embeddings=vocab_size, embedding_dim=d)
        # 2. Feature Embedding (Map column identity -> vector k)
        self.feature_embed = nn.Embedding(num_embeddings=num_features, embedding_dim=k)
        # Lưu lại ID của các cột (0, 1, 2, 3...)
        self.register_buffer('feature_ids', torch.arange(num_features))

    def forward(self, x):
        # x là 1 batch dữ liệu đã được Discretize thành số nguyên
        v_value = self.value_embed(x) 
        v_feature = self.feature_embed(self.feature_ids).unsqueeze(0).expand(x.size(0), -1, -1)
        # 3. Combine: Concatenate dọc theo chiều cuối cùng (v_value || v_feature)
        v_out = torch.cat([v_value, v_feature], dim=-1)
        # 4. Row Representation: Trả về một Ma trận sẵn sàng ném vào lớp Attention
        return v_out

In [ ]:
# ==========================================
# BƯỚC 1: KHỞI TẠO MODEL
# ==========================================
# Cài đặt thông số kiến trúc
NUM_FEATURES = 24    # num_features: Có bao nhiêu cột feature?
VOCAB_SIZE = 51      # vocab_size: Giá trị integer lớn nhất trong data + 1 (chạy từ 0 -> 50)
DIM_VALUE = 8        # d: Độ dài vector d (từ)? (ví dụ 8)
DIM_FEATURE = 4      # k: Độ dài vector k (cột)? (ví dụ 4)

# Khởi tạo (Lúc này máy tính bắt đầu cấp phát RAM để tạo các ma trận Embedding)
tokenizer = TabularTokenizer(num_features=NUM_FEATURES, 
                             vocab_size=VOCAB_SIZE, 
                             d=DIM_VALUE, 
                             k=DIM_FEATURE)

print("Đã tạo xong Model!")

# ==========================================
# BƯỚC 2: CHẠY DỮ LIỆU (Gọi hàm forward)
# ==========================================
# Giả lập một mini-batch gồm 32 dòng log mạng (batch_size = 32), mỗi dòng có 24 tính năng.
# Data bây giờ toàn là số nguyên (từ 0 đến 100) đã rời rạc hóa.
dummy_data_x = torch.randint(low=0, high=50, size=(32, NUM_FEATURES))

print(f"Hình thù Data lúc mới vào: {dummy_data_x.shape} -> (32 dòng, 24 cột số nguyên)")

# Ném data vào model (Nó sẽ tự động chạy hàm forward)
embedded_matrix = tokenizer(dummy_data_x)

# ==========================================
# BƯỚC 3: XEM KẾT QUẢ
# ==========================================
print(f"Hình thù Data sau khi nhúng: {embedded_matrix.shape}")

In [ ]:
class DoW_Classifier(nn.Module):
    def __init__(self, tokenizer, num_features, embedding_dim):
        super().__init__()
        self.tokenizer = tokenizer
        
        # 1. Lớp Transformer Encoder (Trái tim của mô hình)
        # Bác có 24 vector, mỗi vector dài 12 (d_model = 12)
        encoder_layer = nn.TransformerEncoderLayer(d_model=12, nhead=4, batch_first=True)
        # Chồng 2 lớp lên nhau cho máy học sâu hơn
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        
        # 2. Lớp Gom data (Flatten)
        # Biến ma trận 2D thành vector 1D dài thòng lòng (24 * 12 = 288)
        self.flatten = nn.Flatten()

        input_dim = num_features * embedding_dim
        
        # 3. Lớp Đầu ra (Dự đoán)
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1),   # Đầu ra duy nhất 1 con số (Xác suất bot)
        )

    def forward(self, x):
        # Bước 1: Biến số thành ma trận [32, 24, 12]
        embedded = self.tokenizer(x) 
        
        # Bước 2: Cho máy học "sự chú ý" chéo giữa 24 cột
        attended = self.transformer(embedded) 
        
        # Bước 3: Đập dẹp và tính xác suất
        flat = self.flatten(attended)
        out = self.classifier(flat)
        return out

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

print(f"Tổng số tính năng (cột) mang đi đánh trận: {X_train.shape[1]}")

# 1. Ép kiểu sang PyTorch Tensor nhưng GIỮ NGUYÊN Ở CPU (Tuyệt đối không dùng .to(device) ở đây)
X_train_tensor = torch.tensor(X_train).long()
y_train_tensor = torch.tensor(y_train).float().unsqueeze(1)

X_test_tensor = torch.tensor(X_test_dl).long()
y_test_tensor = torch.tensor(y_test_dl).float().unsqueeze(1)

# 2. Tạo Dataset
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# 3. Tạo DataLoader với cấu hình tối ưu
BATCH_SIZE = 4096 # Đẩy lên cao để lấp đầy VRAM

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=4,    # Gọi 4 CPU cores vào bốc data
    pin_memory=True   # Reserve sẵn RAM để đẩy sang VRAM GPU cực nhanh
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=4, 
    pin_memory=True
)

print(f"Đã đóng gói xong DataLoader chuẩn! Tổng số batch: {len(train_loader)}")

In [ ]:
import torch.nn as nn
import torch.optim as optim

# ==========================================
# KHỞI TẠO LẠI MODEL CHO KHỚP DATA THẬT
# ==========================================
NUM_FEATURES = X_train.shape[1] # Lấy tự động số lượng cột
EMBED_DIM = DIM_VALUE + DIM_FEATURE # 8 + 4 = 12
VOCAB_SIZE = int(X_train.max()) + 1 # Tính tự động kích thước từ điển (thường là 51)

print(f"Setup Model: {NUM_FEATURES} features, Vocab size: {VOCAB_SIZE}")

tokenizer = TabularTokenizer(num_features=NUM_FEATURES, vocab_size=VOCAB_SIZE, d=8, k=4)
model = DoW_Classifier(tokenizer, num_features=NUM_FEATURES, embedding_dim=EMBED_DIM).to(device)

# Tự động đếm số lượng hai lớp ngay trên tập Train
# Giả sử y_train của bro là numpy array hoặc PyTorch tensor
num_positive = (y_train == 1).sum() # Số lượng Bot
num_negative = (y_train == 0).sum() # Số lượng Người

# Tính trọng số động
dynamic_pos_weight = num_negative / num_positive
print(f"Trọng số pos_weight tự động: {dynamic_pos_weight:.2f}")

# Ép kiểu về Tensor và ném vào GPU/CPU
weight_tensor = torch.tensor([dynamic_pos_weight], dtype=torch.float32).to(device)

# Gắn vào hàm Loss
criterion = nn.BCEWithLogitsLoss(pos_weight=weight_tensor) 
optimizer = optim.Adam(model.parameters(), lr=0.001) 

EPOCHS = 5 # Chơi 5 epoch train cho lẹ

# ==========================================
# TRAINING THỰC CHIẾN
# ==========================================
print("\n--- BẮT ĐẦU HUẤN LUYỆN ---")
model.train() 

for epoch in range(EPOCHS):
    total_loss = 0
    
    # Lặp qua từng mẻ dữ liệu trong DataLoader
    for batch_X, batch_y in train_loader:

        # Chỉ đẩy từng mẻ nhỏ (mini-batch) lên GPU ngay trước khi tính toán
        batch_X = batch_X.to(device, non_blocking=True)
        batch_y = batch_y.to(device, non_blocking=True)
        
        optimizer.zero_grad()            # 1. Reset đạo hàm
        predictions = model(batch_X)     # 2. Dự đoán
        loss = criterion(predictions, batch_y) # 3. Chấm điểm
        loss.backward()                  # 4. Truy vết lỗi
        optimizer.step()                 # 5. Cập nhật Weights
        
        total_loss += loss.item()
        
    # Tính trung bình Loss của cả Epoch
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}], Điểm Loss Trung Bình: {avg_loss:.4f}")

print("🚀 Huấn luyện xong!")

## Testing: XGBoost (ML) vs Transformer (DL)

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import ConfusionMatrixDisplay
import time

model.eval() # Chuyển sang chế độ kiểm tra
all_preds = []
all_labels = []

# ---------------------------------------------------------
# 1. XGBOOST INFERENCE
# ---------------------------------------------------------
start_time = time.time()
# XGBoost nuốt trọn ma trận Numpy cực nhanh
xgb_preds = xgb_model.predict(X_test_ml) 
xgb_infer_time = time.time() - start_time
print(f"[XGBoost] Thời gian Inference (2 triệu mẫu): {xgb_infer_time:.4f} giây")

# In báo cáo chi tiết
print("--- BÁO CÁO KẾT QUẢ ---")
print(classification_report(y_test_ml, xgb_preds))

ConfusionMatrixDisplay.from_predictions(
    y_test_ml,
    xgb_preds,
    display_labels=["Human", "Bot"]
)

# ---------------------------------------------------------
# 2. TRANSFORMER INFERENCE
# ---------------------------------------------------------
start_time = time.time()
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)
        outputs = model(batch_X)
        # Chuyển xác suất thành nhãn 0 hoặc 1 (ngưỡng 0.8)
        preds = (outputs > 0.8).float()
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch_y.cpu().numpy())
transformer_infer_time = time.time() - start_time
print(f"[Transformer] Thời gian Inference (2 triệu mẫu): {transformer_infer_time:.4f} giây")

# In báo cáo chi tiết
print("--- BÁO CÁO KẾT QUẢ ---")
print(classification_report(all_labels, all_preds))

ConfusionMatrixDisplay.from_predictions(
    all_labels,
    all_preds,
    display_labels=["Human", "Bot"]
)